In [1]:
from tabicl import TabICLClassifier, TabICLRegressor
import nnsight

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import pandas as pd
from huggingface_hub import hf_hub_download
from torch.utils.data import Dataset, DataLoader


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
sonar_dataset = pd.read_csv(
    hf_hub_download('mnemoraorg/sonar-rock-mine','raw_sonar.csv',repo_type="dataset"),
    header = None
)
sonar_dataset

,0,1,2,3,4,5,6,7,8,9,...,51,52,53,54,55,56,57,58,59,60
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,R
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,R
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,R
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117,R
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094,R
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203,0.0187,0.0346,0.0168,0.0177,0.0393,0.1630,0.2028,0.1694,0.2328,0.2684,...,0.0116,0.0098,0.0199,0.0033,0.0101,0.0065,0.0115,0.0193,0.0157,M
204,0.0323,0.0101,0.0298,0.0564,0.0760,0.0958,0.0990,0.1018,0.1030,0.2154,...,0.0061,0.0093,0.0135,0.0063,0.0063,0.0034,0.0032,0.0062,0.0067,M
205,0.0522,0.0437,0.0180,0.0292,0.0351,0.1171,0.1257,0.1178,0.1258,0.2529,...,0.0160,0.0029,0.0051,0.0062,0.0089,0.0140,0.0138,0.0077,0.0031,M
206,0.0303,0.0353,0.0490,0.0608,0.0167,0.1354,0.1465,0.1123,0.1945,0.2354,...,0.0086,0.0046,0.0126,0.0036,0.0035,0.0034,0.0079,0.0036,0.0048,M


In [5]:
# Let's shuffle it
sonar_dataset = sonar_dataset.sample(frac=1.0, random_state=2806)

In [6]:
sonar_dataset

,0,1,2,3,4,5,6,7,8,9,...,51,52,53,54,55,56,57,58,59,60
67,0.0368,0.0403,0.0317,0.0293,0.0820,0.1342,0.1161,0.0663,0.0155,0.0506,...,0.0091,0.0160,0.0160,0.0081,0.0070,0.0135,0.0067,0.0078,0.0068,R
182,0.0095,0.0308,0.0539,0.0411,0.0613,0.1039,0.1016,0.1394,0.2592,0.3745,...,0.0181,0.0019,0.0102,0.0133,0.0040,0.0042,0.0030,0.0031,0.0033,M
117,0.0228,0.0106,0.0130,0.0842,0.1117,0.1506,0.1776,0.0997,0.1428,0.2227,...,0.0098,0.0178,0.0077,0.0074,0.0095,0.0055,0.0045,0.0063,0.0039,M
157,0.0201,0.0178,0.0274,0.0232,0.0724,0.0833,0.1232,0.1298,0.2085,0.2720,...,0.0131,0.0049,0.0104,0.0102,0.0092,0.0083,0.0020,0.0048,0.0036,M
106,0.0331,0.0423,0.0474,0.0818,0.0835,0.0756,0.0374,0.0961,0.0548,0.0193,...,0.0078,0.0174,0.0176,0.0038,0.0129,0.0066,0.0044,0.0134,0.0092,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,0.0180,0.0444,0.0476,0.0698,0.1615,0.0887,0.0596,0.1071,0.3175,0.2918,...,0.0122,0.0114,0.0098,0.0027,0.0025,0.0026,0.0050,0.0073,0.0022,M
58,0.0225,0.0019,0.0075,0.0097,0.0445,0.0906,0.0889,0.0655,0.1624,0.1452,...,0.0034,0.0129,0.0100,0.0044,0.0057,0.0030,0.0035,0.0021,0.0027,R
141,0.0707,0.1252,0.1447,0.1644,0.1693,0.0844,0.0715,0.0947,0.1583,0.1247,...,0.0156,0.0197,0.0135,0.0127,0.0138,0.0133,0.0131,0.0154,0.0218,M
49,0.0119,0.0582,0.0623,0.0600,0.1397,0.1883,0.1422,0.1447,0.0487,0.0864,...,0.0025,0.0103,0.0074,0.0123,0.0069,0.0076,0.0073,0.0030,0.0138,R


In [7]:
# Let's convert the labels to 0/1
sonar_dataset['bin_labels'] = (sonar_dataset[60]=='M') * 1.0
sonar_dataset

,0,1,2,3,4,5,6,7,8,9,...,52,53,54,55,56,57,58,59,60,bin_labels
67,0.0368,0.0403,0.0317,0.0293,0.0820,0.1342,0.1161,0.0663,0.0155,0.0506,...,0.0160,0.0160,0.0081,0.0070,0.0135,0.0067,0.0078,0.0068,R,0.0
182,0.0095,0.0308,0.0539,0.0411,0.0613,0.1039,0.1016,0.1394,0.2592,0.3745,...,0.0019,0.0102,0.0133,0.0040,0.0042,0.0030,0.0031,0.0033,M,1.0
117,0.0228,0.0106,0.0130,0.0842,0.1117,0.1506,0.1776,0.0997,0.1428,0.2227,...,0.0178,0.0077,0.0074,0.0095,0.0055,0.0045,0.0063,0.0039,M,1.0
157,0.0201,0.0178,0.0274,0.0232,0.0724,0.0833,0.1232,0.1298,0.2085,0.2720,...,0.0049,0.0104,0.0102,0.0092,0.0083,0.0020,0.0048,0.0036,M,1.0
106,0.0331,0.0423,0.0474,0.0818,0.0835,0.0756,0.0374,0.0961,0.0548,0.0193,...,0.0174,0.0176,0.0038,0.0129,0.0066,0.0044,0.0134,0.0092,M,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,0.0180,0.0444,0.0476,0.0698,0.1615,0.0887,0.0596,0.1071,0.3175,0.2918,...,0.0114,0.0098,0.0027,0.0025,0.0026,0.0050,0.0073,0.0022,M,1.0
58,0.0225,0.0019,0.0075,0.0097,0.0445,0.0906,0.0889,0.0655,0.1624,0.1452,...,0.0129,0.0100,0.0044,0.0057,0.0030,0.0035,0.0021,0.0027,R,0.0
141,0.0707,0.1252,0.1447,0.1644,0.1693,0.0844,0.0715,0.0947,0.1583,0.1247,...,0.0197,0.0135,0.0127,0.0138,0.0133,0.0131,0.0154,0.0218,M,1.0
49,0.0119,0.0582,0.0623,0.0600,0.1397,0.1883,0.1422,0.1447,0.0487,0.0864,...,0.0103,0.0074,0.0123,0.0069,0.0076,0.0073,0.0030,0.0138,R,0.0


In [8]:
train_size = sonar_dataset.shape[0] // 10 * 7
dev_size = sonar_dataset.shape[0] // 20 * 3
test_size = sonar_dataset.shape[0] - train_size - dev_size
train_size, dev_size, test_size

(140, 30, 38)

In [9]:
train_features = sonar_dataset.iloc[:train_size,:60]
train_labels = sonar_dataset.iloc[:train_size,61]
dev_features = sonar_dataset.iloc[train_size:train_size + dev_size,:60]
dev_labels = sonar_dataset.iloc[train_size:train_size+dev_size,61]
test_features = sonar_dataset.iloc[train_size+dev_size:,:60]
test_labels = sonar_dataset.iloc[train_size+dev_size:,61]

In [10]:
clf = TabICLClassifier(nnsight=True)

In [11]:
clf.fit(train_features, train_labels)


,n_estimators,8
,norm_methods,None
,feat_shuffle_method,'latin'
,class_shuffle_method,'shift'
,outlier_threshold,4.0
,softmax_temperature,0.9
,average_logits,True
,support_many_classes,True
,batch_size,8
,kv_cache,False
,model_path,None


In [12]:
preds = clf.predict(dev_features)
preds

array([1., 0., 0., 1., 0., 0., 1., 1., 1., 1., 1., 0., 1., 0., 1., 0., 0.,
       1., 1., 0., 1., 0., 0., 1., 1., 0., 0., 1., 1., 1.])

In [13]:
dev_labels[16]

np.float64(0.0)

In [14]:
tp,fp,tn, fn = 0, 0, 0, 0
for i,key in enumerate(dev_labels.keys()):
    # print(f"Prediction: {preds[i]}\tLabel: {dev_labels[key]}")
    if preds[i] == 1.0 and dev_labels[key] == 1.0: tp += 1
    if preds[i] == 1.0 and dev_labels[key] == 0.0: fp += 1
    if preds[i] == 0.0 and dev_labels[key] == 1.0: fn += 1
    if preds[i] == 0.0 and dev_labels[key] == 0.0: tn += 1
print(f"Tot instances: {i+1}\tTP: {tp}\tFP: {fp}\tFN: {fn}\tTN: {tn}")



Tot instances: 30	TP: 15	FP: 2	FN: 2	TN: 11


In [15]:
clf.fit(train_features, train_labels)

tracer, X_test_t = clf.get_tracer(dev_features, estimator_idx=0)

with tracer.trace(X_test_t):
    col_emb = tracer.model.col_embedder.output.save()
    row_rep  = tracer.model.row_interactor.output.save()
    out      = tracer.output.save()

print(col_emb.shape)  # (1, train+test, n_features, 128)
print(row_rep.shape)  # (1, train+test, 512)
print(out.shape)      # (1, 30, 2)  — test samples × classes

torch.Size([1, 170, 64, 128])
torch.Size([1, 170, 512])
torch.Size([1, 30, 2])


In [16]:
col_emb

tensor([[[[ 8.4167e-02, -9.4681e-03,  9.5154e-02,  ...,  2.9190e-02,
            8.3008e-02, -1.0559e-01],
          [ 1.7059e-02,  4.0680e-02,  1.2550e-02,  ...,  6.7383e-02,
            3.0167e-02, -1.9054e-03],
          [ 1.2012e-01,  5.0964e-02, -9.6802e-02,  ...,  8.8318e-02,
            1.6284e-01,  2.5234e-03],
          ...,
          [ 5.7129e-01, -1.0947e+00, -4.4141e-01,  ..., -1.3855e-01,
           -3.6279e-01,  7.7246e-01],
          [ 9.7070e-01, -1.1309e+00, -8.8574e-01,  ..., -5.4932e-01,
            3.2422e-01,  3.2227e-02],
          [ 9.7510e-01, -1.5566e+00, -1.5796e-01,  ..., -7.7246e-01,
           -1.3062e-01,  4.4360e-01]],

         [[ 8.4167e-02, -9.4681e-03,  9.5154e-02,  ...,  2.9190e-02,
            8.3008e-02, -1.0559e-01],
          [ 1.7059e-02,  4.0680e-02,  1.2550e-02,  ...,  6.7383e-02,
            3.0167e-02, -1.9054e-03],
          [ 1.2012e-01,  5.0964e-02, -9.6802e-02,  ...,  8.8318e-02,
            1.6284e-01,  2.5234e-03],
          ...,
     